# 3D MNIST Original Mesh Visualization
This notebook visualizes the original generated meshes from `src/`.

In [1]:
import os
import glob
import plotly.graph_objects as go
import numpy as np

# Get original .obj files
obj_files = sorted(glob.glob("src/train/*.obj"))[:5]

for file in obj_files:
    vertices = []
    faces = []
    with open(file, 'r') as f:
        for line in f:
            if line.startswith("v "):
                vertices.append([float(x) for x in line.split()[1:]])
            elif line.startswith("f "):
                faces.append([int(x)-1 for x in line.split()[1:]])
                
    vert = np.array(vertices)
    face = np.array(faces)
    
    label = os.path.basename(file).split('_')[0]
    
    fig = go.Figure(data=[
        go.Mesh3d(
            x=vert[:, 0],
            y=vert[:, 1],
            z=vert[:, 2],
            i=face[:, 0],
            j=face[:, 1],
            k=face[:, 2],
            color='lightpink',
            opacity=1.0,
            name=f'Label: {label}',
            lighting=dict(ambient=0.4, diffuse=0.8, specular=0.2),
        )
    ])
    fig.update_layout(
        title=f'Original Mesh - Label: {label}',
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data'
        )
    )
    fig.show()


# 3D MNIST Reconstructed Mesh Visualization
This section reconstructs meshes dynamically from the offline generated `.npz` sparse voxel archives.

In [2]:
import torch
import conquer3d as c3d

# Get the first 5 .npz files from the sdf/train directory
npz_files = sorted(glob.glob("sdf/train/*.npz"))[:5]

meshes = []
for file in npz_files:
    data = np.load(file)
    idx_grids = data['idx_grids']
    sdf = data['sdf']
    
    # 1. Initialize full dense grid with positive 10.0 (outside geometry)
    dense_sdf = torch.full((32*32*32,), 10.0, device="cuda")
    
    # 2. Pad [N, 3] coordinates with a Batch Index column to get [N, 4]
    N = idx_grids.shape[0]
    sparse_coords = torch.zeros((N, 4), dtype=torch.int32)
    sparse_coords[:, 1:] = torch.tensor(idx_grids, dtype=torch.int32)
    
    # 3. Use sparse2voxel to map the [x,y,z] indices directly to a flattened 1D array index
    vertex_1d = c3d.data_structure.sparse2voxel(sparse_coords, res=[33, 33, 33]).to("cuda")
    
    # 4. Map the sparse active SDF features into the dense grid
    dense_sdf[vertex_1d] = torch.tensor(sdf, device="cuda")
    
    # 5. Create Voxel Grid spatial constraints
    grid_vertices, voxels, _ = c3d.data_structure.create_voxel_grid(
        [-1.2, -0.6, -1.2], [1.2, 0.6, 1.2], [32, 32, 32], device="cuda"
    )
    
    # 6. Reconstruct dense mesh via Marching Cubes on GPU
    vertices, faces, _, _ = c3d.ops.marching_cubes(
        grid_vertices=grid_vertices,
        voxels=voxels,
        voxel_values=dense_sdf,
        iso=0.0
    )
    meshes.append((vertices.cpu().numpy(), faces.cpu().numpy(), file))


In [3]:
for vert, face, file in meshes:
    # Parse label from filename (e.g. 5_1234.npz -> 5)
    label = os.path.basename(file).split('_')[0]
    
    fig = go.Figure(data=[
        go.Mesh3d(
            x=vert[:, 0],
            y=vert[:, 1],
            z=vert[:, 2],
            i=face[:, 0],
            j=face[:, 1],
            k=face[:, 2],
            color='lightblue',
            opacity=1.0,
            name=f'Label: {label}',
            lighting=dict(ambient=0.4, diffuse=0.8, specular=0.2),
        )
    ])
    fig.update_layout(
        title=f'Reconstructed Mesh from Sparse Voxel - Label: {label}',
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data'
        )
    )
    fig.show()
